# 04 — Evaluation

Loads the trained model and evaluates it on the test set using accuracy, precision, recall, F1-score, and a confusion matrix.

## Imports

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
DATA_DIR = '/content/drive/MyDrive/datasets/catsvsdogs' if IN_COLAB else '../data/raw'

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 16 * 16, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

class CatsDogsDataset(Dataset):
    def __init__(self, csv_path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, torch.tensor(row['label'], dtype=torch.float32)

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

splits_dir = Path(DATA_DIR) / 'splits'
test_loader = DataLoader(CatsDogsDataset(splits_dir / 'test.csv', transform), batch_size=32)

## Load Model

In [ ]:
model = CNN().to(device)
model.load_state_dict(torch.load(Path(DATA_DIR) / 'models' / 'cnn.pth', map_location=device))
model.eval()
print('Model loaded.')

## Run Inference on Test Set

In [ ]:
all_preds, all_labels = [], []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out = model(imgs)
        preds = (torch.sigmoid(out) >= 0.5).cpu().int().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.int().tolist())

## Classification Report (Accuracy / Precision / Recall / F1)

In [ ]:
print(classification_report(all_labels, all_preds, target_names=['Cat', 'Dog']))

## Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', xticklabels=['Cat', 'Dog'], yticklabels=['Cat', 'Dog'], cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## Findings & Observations

- **Accuracy**: 
- **Loss**: 
- **Observations**: 